<a href="https://colab.research.google.com/github/mkwonghsni/bengkel_daml_2026/blob/main/hari_kedua/linear_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

import torch
from torch import nn

In [ ]:
sns.set_theme(context="notebook", style="darkgrid")

# Menjana Data

In [ ]:
rng = np.random.default_rng()

In [ ]:
x = np.linspace(10, 100, 20)
X = x + rng.normal(0, 0.01, len(x))
y = x + rng.normal(0, 1, len(x))

# Fungsi

In [ ]:
def melatih_model_nn(model, loss_fx, optimizer, X_latih, y_latih):
    model.train()
    y_ramal = model(X_latih.unsqueeze(1))
    loss = loss_fx(y_ramal, y_latih.unsqueeze(1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss

In [ ]:
def menguji_model_nn(model, loss_fx, X_ujian, y_ujian):
    model.eval()
    with torch.inference_mode():
        y_ramal = model(X_ujian.unsqueeze(1))
        loss = loss_fx(y_ramal, y_ujian.unsqueeze(1))
    return loss

# EDA
1. Meneliti data - plot X vs y
2. Memahami shape data

In [ ]:
X, y

In [ ]:
X.shape, y.shape

In [ ]:
sns.scatterplot(x=X, y=y)
plt.xlabel("X")
plt.ylabel("y")
plt.show()

# Regression Analysis - Conventional
1. Menggunakan kaedah statistik melalui sklearn linear regression
2. Membuat ramalan nilai y_skl_ramal
3. Mengira ralat model dengan menggunakan metrik mean absolute error
4. Memplot graf y vs y_skl_ramal

In [ ]:
model_skl_lr = LinearRegression()
model_skl_lr.fit(X.reshape(-1, 1), y)

# y = mX + c
m = model_skl_lr.coef_
c = model_skl_lr.intercept_

# ramalan
y_skl_ramal = model_skl_lr.predict(X.reshape(-1, 1))

# ralat model - MAE
mae_skl_lr = mean_absolute_error(y, y_skl_ramal)

In [ ]:
sns.scatterplot(x=X, y=y, label="asal")
sns.scatterplot(x=X, y=y_skl_ramal, label="ramalan")

plt.text(x=20, y=80, s=f'y = {c.item():.4f} + {m.item():.4f}X')
plt.text(x=20, y=70, s=f'MAE = {mae_skl_lr:.4f}')

plt.xlabel("X")
plt.ylabel("y")
plt.show()

# Regression Analysis - Neural Network
1. Menukarkan data daripada numpy.array kepada torch.tensor
2. Mengasingkan data kepada latihan dan ujian
3. Meneliti data - plot data latihan vs ujian
4. Membina model NN
5. Melatih model
6. Menilai model

In [ ]:
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

In [ ]:
X.shape, y.shape

In [ ]:
X_latih, X_ujian, y_latih, y_ujian = train_test_split(X, y, test_size=0.2)

In [ ]:
sns.scatterplot(x=X_latih, y=y_latih, label="Latihan")
sns.scatterplot(x=X_ujian, y=y_ujian, label="Ujian")

plt.xlabel("X")
plt.ylabel("y")
plt.show()

In [ ]:
model_nn = nn.Sequential(
    nn.Linear(1, 1),
)

In [ ]:
loss_fx = nn.L1Loss()
optimizer = torch.optim.SGD(model_nn.parameters(), lr=0.001)

In [ ]:
epochs = 100

for epoch in range(epochs):
    latih_loss = melatih_model_nn(model_nn, loss_fx, optimizer, X_latih, y_latih)
    ujian_loss = menguji_model_nn(model_nn, loss_fx, X_ujian, y_ujian)

    if epoch % 10 == 0:
        print(f'Epoch: {epoch} | Loss latihan: {latih_loss:.4f} | Loss ujian: {ujian_loss:.4f}')

In [ ]:
model_nn.eval()
with torch.inference_mode():
    y_pred = model_nn(X_ujian.unsqueeze(1))

mae_nn = mean_absolute_error(y_ujian, y_pred)

In [ ]:
sns.scatterplot(x=X_latih, y=y_latih, label="Latihan")
sns.scatterplot(x=X_ujian, y=y_ujian, label="Ujian")
sns.scatterplot(x=X_ujian, y=y_pred.squeeze(), label="Ramalan")

plt.text(x=20, y=70, s=f'MAE = {mae_nn:.4f}')
plt.xlabel("X")
plt.ylabel("y")
plt.show()

# Perbandingan Kaedah Conventional vs Neural Network

In [ ]:
sns.scatterplot(x=X, y=y, label="Asal")
sns.scatterplot(x=X, y=y_skl_ramal, label="SKL")
sns.scatterplot(x=X_ujian, y=y_pred.squeeze(), label="NN")

plt.text(20, 70, f'MAE SKL = {mae_skl_lr:.4f}')
plt.text(20, 60, f'MAE NN = {mae_nn:.4f}')

plt.xlabel("X")
plt.ylabel("y")
plt.show()